# Obfuscation-Utility Trade-Off Analysis

This notebook systematically varies obfuscation parameters (`patch_size` and `group_size`) and measures
the resulting trade-off between task accuracy and privacy (attack resistance).

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "vit_obfuscation").exists():
            return path
    raise RuntimeError("Could not find repository root. Start Jupyter inside the repository.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_DIR = ROOT / "configs" / "experiments"
CANONICAL_OUTPUT_DIR = ROOT / "outputs" / "revision_v3"
NOTEBOOK_OUTPUT_DIR = ROOT / "outputs" / "notebook_runs"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Canonical artifacts: {CANONICAL_OUTPUT_DIR}")
print(f"Notebook outputs: {NOTEBOOK_OUTPUT_DIR}")

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
# Suppress noisy HTTP logs
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Configuration

Select a base experiment to sweep over. CIFAR-10 classification provides fast iteration.

In [ ]:
from vit_obfuscation.config.experiment import ExperimentConfig

config = ExperimentConfig.from_yaml(str(CONFIG_DIR / "vit_cifar10.yaml"))
config.output_dir = str(NOTEBOOK_OUTPUT_DIR)
print(f"Base experiment: {config.name}")
print(f"Current patch_size: {config.obfuscation.patch_size}")
print(f"Current group_size: {config.obfuscation.group_size}")

## Step 2: Patch Size Sweep

Vary `patch_size` while keeping `group_size` fixed.

In [ ]:
from vit_obfuscation.analysis.tradeoff import run_tradeoff_sweep

patch_sizes = [7, 14, 16, 28]
patch_sweep = run_tradeoff_sweep(config, "patch_size", patch_sizes)

import pandas as pd
df_patch = pd.DataFrame([{
    "patch_size": p.param_value,
    "Clean Accuracy": f"{p.clean_accuracy:.4f}",
    "Obfuscated Accuracy": f"{p.obfuscated_accuracy:.4f}",
    "Attack SSIM": f"{p.attack_ssim:.4f}",
    "Attack PSNR (dB)": f"{p.attack_psnr:.2f}",
} for p in patch_sweep.points])
print(df_patch.to_string(index=False))

## Step 3: Group Size Sweep

Vary `group_size` while keeping `patch_size` fixed.

In [ ]:
group_sizes = [1, 10, 50, 100, 200]
group_sweep = run_tradeoff_sweep(config, "group_size", group_sizes)

df_group = pd.DataFrame([{
    "group_size": p.param_value,
    "Clean Accuracy": f"{p.clean_accuracy:.4f}",
    "Obfuscated Accuracy": f"{p.obfuscated_accuracy:.4f}",
    "Attack SSIM": f"{p.attack_ssim:.4f}",
    "Attack PSNR (dB)": f"{p.attack_psnr:.2f}",
} for p in group_sweep.points])
print(df_group.to_string(index=False))

## Step 4: Pareto Plot

Visualize the accuracy vs. privacy trade-off. Each point represents a different parameter value.
The ideal operating point is in the upper-left (high accuracy, low attack SSIM).

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: patch_size sweep
patch_acc = [p.obfuscated_accuracy for p in patch_sweep.points]
patch_ssim = [p.attack_ssim for p in patch_sweep.points]
patch_labels = [p.param_value for p in patch_sweep.points]
ax1.scatter(patch_acc, patch_ssim, s=100, zorder=5)
for i, label in enumerate(patch_labels):
    ax1.annotate(f"patch_size={label}", (patch_acc[i], patch_ssim[i]),
                 textcoords="offset points", xytext=(8, 8), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))
ax1.set_xlabel("Obfuscated Accuracy")
ax1.set_ylabel("Attack SSIM")
ax1.set_title("Patch Size Sweep: Accuracy vs. Privacy")

# Right: group_size sweep
group_acc = [p.obfuscated_accuracy for p in group_sweep.points]
group_ssim = [p.attack_ssim for p in group_sweep.points]
group_labels = [p.param_value for p in group_sweep.points]
ax2.scatter(group_acc, group_ssim, s=100, zorder=5, color="tab:orange")
for i, label in enumerate(group_labels):
    ax2.annotate(f"group_size={label}", (group_acc[i], group_ssim[i]),
                 textcoords="offset points", xytext=(8, 8), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))
ax2.set_xlabel("Obfuscated Accuracy")
ax2.set_ylabel("Attack SSIM")
ax2.set_title("Group Size Sweep: Accuracy vs. Privacy")

plt.tight_layout()
plt.show()

## Step 5: Visual Comparison Across Parameters

Show the same image obfuscated with different parameter settings.

In [ ]:
from datasets import load_dataset
from vit_obfuscation.obfuscation.obfuscator import ChannelWisePatchLevelObfuscator
from torchvision import transforms

# Load one sample image.
ds = load_dataset(config.dataset.hf_dataset_name_or_path, split=config.dataset.eval_split)
sample_image = ds[0][config.dataset.input_column]
if hasattr(sample_image, "convert"):
    sample_image = sample_image.convert("RGB")

image_size = 224
num_channels = 3
transform = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
img_tensor = transform(sample_image).unsqueeze(0)

fig, axes = plt.subplots(1, len(patch_sizes) + 1, figsize=(4 * (len(patch_sizes) + 1), 4))

# Original
axes[0].imshow(img_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1))
axes[0].set_title("Original")
axes[0].axis("off")

# Obfuscated with different patch_sizes
for i, ps in enumerate(patch_sizes):
    obfuscator = ChannelWisePatchLevelObfuscator(
        image_size=image_size,
        num_channels=num_channels,
        patch_size=ps,
        group_size=config.obfuscation.group_size,
        apply_patch_permutation=config.obfuscation.apply_patch_permutation,
    )
    with torch.no_grad():
        obf_img = obfuscator(img_tensor)
    obf_img = (obf_img - obf_img.min()) / (obf_img.max() - obf_img.min() + 1e-8)
    axes[i + 1].imshow(obf_img.squeeze(0).permute(1, 2, 0).clamp(0, 1))
    axes[i + 1].set_title(f"patch_size={ps}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

## Discussion

The trade-off analysis reveals the Pareto frontier between task utility and privacy protection.
The default parameters (`patch_size=14`, `group_size=100`) represent a balanced operating point
that achieves empirical privacy protection while maintaining competitive task accuracy.